# 📊 Análisis de Batches - Entrega 3

Código para calcular métricas de los 4 batches y generar análisis comparativo.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.style.use('default')
sns.set_palette("husl")

print("✅ Librerías importadas")

✅ Librerías importadas


## Función para calcular métricas

In [2]:
def calcular_metricas_batch(batch_id, pred_file_name):
    """
    Calcula Precision, Recall y F1-Score comparando predicciones vs datos reales.
    
    Args:
        batch_id (int): Número del batch (1, 2, 3, 4)
        pred_file_name (str): Nombre del archivo de predicciones
        
    Returns:
        dict: Diccionario con métricas y conteos
    """
    print(f"\n{'='*70}")
    print(f"📊 BATCH {batch_id}")
    print(f"{'='*70}")
    
    # Cargar datos reales
    real_file = Path('data') / f'batch_t{batch_id}.parquet'
    real_df = pd.read_parquet(real_file)
    real_pairs = set(
        map(tuple, real_df[['customer_id', 'product_id']].drop_duplicates().values)
    )
    print(f"✅ Compras reales: {len(real_pairs):,} pares únicos")
    
    # Cargar predicciones
    pred_file = Path('predictions') / pred_file_name
    pred_df = pd.read_csv(pred_file, names=['customer_id', 'product_id'])
    pred_pairs = set(map(tuple, pred_df.values))
    print(f"✅ Predicciones: {len(pred_pairs):,} pares")
    
    # Calcular TP, FP, FN
    tp = len(pred_pairs & real_pairs)  # Intersección
    fp = len(pred_pairs - real_pairs)  # Predichos pero no reales
    fn = len(real_pairs - pred_pairs)  # Reales pero no predichos
    
    print(f"\n📈 Conteos:")
    print(f"   TP (Aciertos):               {tp:,}")
    print(f"   FP (Falsos positivos):       {fp:,}")
    print(f"   FN (Falsos negativos):       {fn:,}")
    
    # Calcular métricas
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f"\n📊 Métricas:")
    print(f"   Precision: {precision:.4f}")
    print(f"   Recall:    {recall:.4f}")
    print(f"   F1-Score:  {f1:.4f}")
    
    # Métricas adicionales
    cobertura = recall
    tasa_predicciones = len(pred_pairs) / len(real_pairs) if len(real_pairs) > 0 else 0
    
    print(f"\n💼 Negocio:")
    print(f"   Cobertura (% compras capturadas): {cobertura:.2%}")
    print(f"   Predicciones por compra real:     {tasa_predicciones:.2f}x")
    
    return {
        'batch_id': batch_id,
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'cobertura': cobertura,
        'tasa_predicciones': tasa_predicciones,
        'num_predicciones': len(pred_pairs),
        'num_reales': len(real_pairs)
    }

print("✅ Función definida")

✅ Función definida


## Calcular métricas para cada batch

In [3]:
# Batch 1
metricas_b1 = calcular_metricas_batch(1, 'customer_product_sample.csv')

# Batch 2
metricas_b2 = calcular_metricas_batch(2, 'customer_product_sample 2.csv')

# Batch 3
metricas_b3 = calcular_metricas_batch(3, 'customer_product_sample 3.csv')

# Batch 4
metricas_b4 = calcular_metricas_batch(4, 'customer_product_sample 4.csv')


📊 BATCH 1
✅ Compras reales: 3,264 pares únicos
✅ Predicciones: 3,264 pares

📈 Conteos:
   TP (Aciertos):               3,264
   FP (Falsos positivos):       0
   FN (Falsos negativos):       0

📊 Métricas:
   Precision: 1.0000
   Recall:    1.0000
   F1-Score:  1.0000

💼 Negocio:
   Cobertura (% compras capturadas): 100.00%
   Predicciones por compra real:     1.00x

📊 BATCH 2
✅ Compras reales: 5,103 pares únicos
✅ Predicciones: 5,798 pares

📈 Conteos:
   TP (Aciertos):               2,559
   FP (Falsos positivos):       3,239
   FN (Falsos negativos):       2,544

📊 Métricas:
   Precision: 0.4414
   Recall:    0.5015
   F1-Score:  0.4695

💼 Negocio:
   Cobertura (% compras capturadas): 50.15%
   Predicciones por compra real:     1.14x

📊 BATCH 3
✅ Compras reales: 4,922 pares únicos
✅ Predicciones: 3,264 pares

📈 Conteos:
   TP (Aciertos):               1,701
   FP (Falsos positivos):       1,563
   FN (Falsos negativos):       3,221

📊 Métricas:
   Precision: 0.5211
   Recall:    0.3

## Consolidar resultados

In [ ]:
# Crear DataFrame con todas las métricas
df_metricas = pd.DataFrame([metricas_b1, metricas_b2, metricas_b3, metricas_b4])

print("\n" + "="*80)
print("📊 RESUMEN CONSOLIDADO")
print("="*80)
print(df_metricas[['batch_id', 'precision', 'recall', 'f1', 'tp', 'fp', 'fn', 
                     'num_predicciones', 'num_reales']].to_string(index=False))
print("="*80)

## Gráficos de evolución

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

batches = df_metricas['batch_id'].values

# Gráfico 1: F1-Score
axes[0, 0].plot(batches, df_metricas['f1'], marker='o', linewidth=2, markersize=8, color='#d62728')
axes[0, 0].set_title('Evolución del F1-Score', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Batch')
axes[0, 0].set_ylabel('F1-Score')
axes[0, 0].set_xticks(batches)
axes[0, 0].grid(True, alpha=0.3)
for i, v in enumerate(df_metricas['f1']):
    axes[0, 0].text(batches[i], v + 0.01, f'{v:.4f}', ha='center', fontsize=9)

# Gráfico 2: Precision vs Recall
axes[0, 1].plot(batches, df_metricas['precision'], marker='s', label='Precision', linewidth=2, markersize=8)
axes[0, 1].plot(batches, df_metricas['recall'], marker='^', label='Recall', linewidth=2, markersize=8)
axes[0, 1].set_title('Precision y Recall', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Batch')
axes[0, 1].set_ylabel('Valor')
axes[0, 1].set_xticks(batches)
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Gráfico 3: TP, FP, FN
width = 0.25
x = np.arange(len(batches))
axes[1, 0].bar(x - width, df_metricas['tp'], width, label='TP', color='#2ca02c')
axes[1, 0].bar(x, df_metricas['fp'], width, label='FP', color='#ff7f0e')
axes[1, 0].bar(x + width, df_metricas['fn'], width, label='FN', color='#d62728')
axes[1, 0].set_title('TP, FP, FN por Batch', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Batch')
axes[1, 0].set_ylabel('Cantidad')
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(batches)
axes[1, 0].legend()
axes[1, 0].grid(axis='y', alpha=0.3)

# Gráfico 4: Predicciones vs Reales
axes[1, 1].bar(batches, df_metricas['num_predicciones'], alpha=0.6, label='Predicciones', color='#1f77b4')
axes[1, 1].bar(batches, df_metricas['num_reales'], alpha=0.6, label='Compras reales', color='#2ca02c')
axes[1, 1].set_title('Predicciones vs Compras Reales', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Batch')
axes[1, 1].set_ylabel('Cantidad')
axes[1, 1].set_xticks(batches)
axes[1, 1].legend()
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Análisis estadístico

In [ ]:
print("\n" + "="*80)
print("📊 ESTADÍSTICAS")
print("="*80)

print(f"\n🎯 F1-Score:")
print(f"   Promedio: {df_metricas['f1'].mean():.4f}")
print(f"   Mínimo:   {df_metricas['f1'].min():.4f} (Batch {int(df_metricas.loc[df_metricas['f1'].idxmin(), 'batch_id'])})")
print(f"   Máximo:   {df_metricas['f1'].max():.4f} (Batch {int(df_metricas.loc[df_metricas['f1'].idxmax(), 'batch_id'])})")
print(f"   Desv. Est: {df_metricas['f1'].std():.4f}")

print(f"\n📈 Variación temporal:")
print(f"   Mejora B1→B4: {((df_metricas.iloc[-1]['f1'] / df_metricas.iloc[0]['f1']) - 1) * 100:+.2f}%")

print(f"\n💼 Totales:")
print(f"   Predicciones totales: {df_metricas['num_predicciones'].sum():,}")
print(f"   Compras reales totales: {df_metricas['num_reales'].sum():,}")
print(f"   TP totales: {df_metricas['tp'].sum():,}")
print(f"   FP totales: {df_metricas['fp'].sum():,}")
print(f"   FN totales: {df_metricas['fn'].sum():,}")

print("\n" + "="*80)

## Exportar resultados

In [ ]:
df_metricas.to_csv('metricas_consolidadas.csv', index=False)
print("✅ Métricas exportadas a 'metricas_consolidadas.csv'")